<a href="https://colab.research.google.com/github/dejanjovic1283-ui/titanic-survival-prediction/blob/main/titanic_kaggle_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Required Libraries

In this step, we install all necessary Python libraries for the project, including pandas for data manipulation and scikit-learn for machine learning.

In [8]:
# Install required libraries
!pip install -q kaggle pandas scikit-learn matplotlib seaborn

## Upload Kaggle API Token

We upload the Kaggle API token (`kaggle.json`) to authenticate and access datasets programmatically.

In [9]:
# Upload kaggle.json file
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"dejan83","key":"4d7af9cae581ea0711e77de69825c307"}'}

## Configure Kaggle API

We move the Kaggle API token to the correct directory and set permissions.

In [10]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API configured successfully")

Kaggle API configured successfully


## Download Titanic Dataset

We download the Titanic dataset directly from Kaggle.

In [11]:
# Download Titanic dataset from Kaggle
!kaggle competitions download -c titanic

# Unzip dataset
!unzip -o titanic.zip

100% 34.1k/34.1k [00:00<00:00, 49.2MB/s]

Archive:  titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


## Load Dataset

We load the training and test datasets into pandas DataFrames.

In [12]:
import pandas as pd

# Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Display shapes
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Data Exploration

We explore the dataset structure, data types, and summary statistics.

In [13]:
# Check data info
train_df.info()

# Summary statistics
train_df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Dooley, Mr. Patrick",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


## Data Cleaning and Preprocessing

We handle missing values, remove unnecessary columns, and convert categorical variables into numerical format.

In [14]:
# Copy datasets
df_train = train_df.copy()
df_test = test_df.copy()

# Save PassengerId for submission
test_passenger_ids = df_test["PassengerId"]

# Drop unnecessary columns
drop_cols = ["Name", "Ticket", "Cabin"]
df_train = df_train.drop(columns=drop_cols)
df_test = df_test.drop(columns=drop_cols)

# Fill missing values
df_train["Age"] = df_train["Age"].fillna(df_train["Age"].median())
df_test["Age"] = df_test["Age"].fillna(df_train["Age"].median())

df_train["Embarked"] = df_train["Embarked"].fillna(df_train["Embarked"].mode()[0])

df_train["Fare"] = df_train["Fare"].fillna(df_train["Fare"].median())
df_test["Fare"] = df_test["Fare"].fillna(df_train["Fare"].median())

# Convert categorical variables to numeric
df_train = pd.get_dummies(df_train, columns=["Sex", "Embarked"], drop_first=True)
df_test = pd.get_dummies(df_test, columns=["Sex", "Embarked"], drop_first=True)

# Align columns
df_train, df_test = df_train.align(df_test, join="left", axis=1, fill_value=0)

df_train.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,22.0,1,0,7.2500,True,False,True
1,2,1,1,38.0,1,0,71.2833,False,False,False
2,3,1,3,26.0,0,0,7.9250,False,False,True
3,4,1,1,35.0,1,0,53.1000,False,False,True
4,5,0,3,35.0,0,0,8.0500,True,False,True


## Define Features and Target

We separate input features (X) and target variable (y).

In [15]:
# Define features and target
X = df_train.drop("Survived", axis=1)
y = df_train["Survived"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 9)
y shape: (891,)


## Train/Test Split

We split the dataset into training and validation sets.

In [16]:
from sklearn.model_selection import train_test_split

# Split data
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_valid.shape)

(712, 9) (179, 9)


## Train Model

We train a Logistic Regression model on the dataset.

In [17]:
from sklearn.linear_model import LogisticRegression

# Initialize model
model = LogisticRegression(max_iter=1000)

# Train model
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Model Evaluation

We evaluate the model using accuracy, classification report, and confusion matrix.

In [18]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predictions
y_pred = model.predict(X_valid)

# Accuracy
accuracy = accuracy_score(y_valid, y_pred)
print("Validation accuracy:", accuracy)

# Detailed metrics
print("\nClassification Report:\n")
print(classification_report(y_valid, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_valid, y_pred))

Validation accuracy: 0.8100558659217877

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.89      0.85       110
           1       0.80      0.68      0.73        69

    accuracy                           0.81       179
   macro avg       0.81      0.79      0.79       179
weighted avg       0.81      0.81      0.81       179


Confusion Matrix:

[[98 12]
 [22 47]]


## Create Submission File

We generate predictions for the test dataset and create a submission file for Kaggle.

In [23]:
# Remove target column from test set if it exists
if "Survived" in df_test.columns:
    df_test = df_test.drop("Survived", axis=1)

print(df_test.columns)

Index(['PassengerId', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male',
       'Embarked_Q', 'Embarked_S'],
      dtype='object')


In [24]:
# Predict on test set
test_predictions = model.predict(df_test)

# Create submission DataFrame
submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": test_predictions.astype(int)
})

# Save file
submission.to_csv("submission.csv", index=False)

print("Submission file created successfully")

Submission file created successfully


## Download Submission

We download the submission file to upload it on Kaggle.

In [25]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>